In [ ]:
%load_ext cudf.pandas

In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:

#
import pandas as pd
import numpy as np
from pathlib import Path
from utils.benchmarks import BENCHMARKS_TO_PATHS

In [ ]:
### cell 0 ###

benchmark_name = "imdb"
filename = Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "movie_metadata.csv"

m = pd.read_csv(filename)
factor = 500
m = pd.concat([m] * factor)

In [ ]:
### cell 1 ###

m.info()

In [ ]:
### cell 2 ###

m.movie_title = m.movie_title.str.strip()
m.duration = pd.to_numeric(m.duration)
m.budget = pd.to_numeric(m.budget)
m.gross = pd.to_numeric(m.gross)
m.imdb_score = pd.to_numeric(m.imdb_score)
m.set_index("movie_title", inplace=True)
m.drop(
    [
        "color",
        "director_facebook_likes",
        "actor_3_facebook_likes",
        "actor_2_name",
        "actor_1_facebook_likes",
        "actor_1_name",
    ],
    axis=1,
    inplace=True,
)
m.drop(
    [
        "cast_total_facebook_likes",
        "movie_imdb_link",
        "language",
        "actor_2_facebook_likes",
        "aspect_ratio",
    ],
    axis=1,
    inplace=True,
)
m.drop(
    ["actor_3_name", "facenumber_in_poster", "plot_keywords", "country"],
    axis=1,
    inplace=True,
)

In [ ]:
### cell 3 ###

m.columns

In [ ]:
### cell 4 ###

# number of movies longer than 90 min (GPU‐accelerated sum)
ninety_min_num_movies = (m.duration > 90.0).sum()

# total number of movies (include NaNs the same way as the original mask m.duration != np.nan did)
total_num_movies = len(m.duration)

ninety_min_num_movies / total_num_movies

In [ ]:
### cell 5 ###

two_hour_movies = len(m.duration[m.duration > 120.0])
two_hour_movies / total_num_movies

In [ ]:
### cell 6 ###
# pull the director column once to reduce repeated attribute lookups
director = m['director_name']
# the original `!= np.nan` mask never actually drops any rows (NaN!=NaN is True),
# so num_movies_directed should be the total length, not the non‐null count
d = director
num_movies_directed = len(d)
# count Spielberg films on the GPU
spiel = (d == 'Steven Spielberg').sum()
# ratio as a Python float
spiel / num_movies_directed

In [ ]:
### cell 7 ###

e_movies = m[m.director_name == "Clint Eastwood"]
e_gross_under_budget = len(
    e_movies[
        (e_movies.gross != np.nan)
        & (e_movies.budget != np.nan)
        & (e_movies.gross < e_movies.budget)
    ]
)
e_gross_under_budget / len(e_movies.index)

In [ ]:
### cell 8 ###

# Filter out rows with null gross or budget on the GPU
valid = m.dropna(subset=["gross", "budget"])

# Compute the fraction where gross exceeds budget (all on GPU)
ratio = (valid.gross > valid.budget).mean()

ratio

In [ ]:
### cell 9 ###

average_gross = m.gross.mean()
movie_grossed_over_average = (m.gross > average_gross).sum()
total_movie_with_gross = m.gross.count()
movie_grossed_over_average / total_movie_with_gross

In [ ]:
### cell 10 ###

# Optimized GPU-friendly version
movies_with_scores = m.dropna(subset=["imdb_score", "gross", "budget"])[["imdb_score", "gross", "budget"]]
# build boolean mask for positive IMDb scores
mask_pos = movies_with_scores["imdb_score"] > 6
# compute false-positives and positive counts entirely on the GPU
false_positive_rate = ((movies_with_scores["gross"] < movies_with_scores["budget"]) & mask_pos).sum() / mask_pos.sum()
false_positive_rate

In [ ]:
### cell 11 ###

# Vectorized GPU-based computation
def_mask = movies_with_scores.imdb_score.le(6)
numerator = (def_mask & movies_with_scores.gross.gt(movies_with_scores.budget)).sum()
denominator = def_mask.sum()
numerator / denominator

In [ ]:
### cell 12 ###

scores = m.imdb_score

# plt.figure(figsize=(10, 3))
# plt.hist(scores, bins=np.arange(1, 11))
# plt.title("Distribution of Ratings")
# plt.xlabel("IMDB Rating")
# plt.ylabel("# of Movies")
# plt.show()

In [ ]:
### cell 13 ###

scores.describe()

In [ ]:
### cell 14 ###

mean = scores.mean()
median = scores.quantile(0.5)
std = scores.std()
mean, median, std